# 1 · Preprocesamiento de datos

> **Tipo de ML:** `{{ ml_type }}`

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import joblib

from {{ project_slug }}.data.make_dataset import load_data
from {{ project_slug }}.features.build_features import preprocess_data
from {{ project_slug }}.utils.paths import PROCESSED_DATA_DIR, ARTIFACTS_DIR


## 2. Cargar datos crudos

In [ ]:
DATA_FILE = '<nombre>.csv'   # ← pon aquí la ruta a tu fichero
try:
    df = load_data(DATA_FILE)
except FileNotFoundError:
    raise FileNotFoundError(
        f"Archivo no encontrado: {DATA_FILE}\n"
        "Coloca el CSV en data/raw/ y ajusta DATA_FILE."
    )
print(df.shape)

In [ ]:
df.head()

## 3. Preprocesar

{% if ml_type in ['supervisado', 'redes_neuronales'] %}
Indica la columna objetivo (`TARGET_COL`) y el tipo de scaler.
{% elif ml_type == 'no_supervisado' %}
No se necesita columna objetivo.
{% endif %}

In [ ]:
{% if ml_type in ['supervisado', 'redes_neuronales'] %}
TARGET_COL = '<columna_objetivo>'   # ← ajusta
SCALER_TYPE = 'standard'            # 'standard' | 'minmax'
X_train, X_test, y_train, y_test = preprocess_data(df, target_col=TARGET_COL, scaler_type=SCALER_TYPE)

print('X_train:', X_train.shape)
print('X_test: ', X_test.shape)
print('Balance clases (train):', y_train.value_counts(normalize=True).to_dict())
{% elif ml_type == 'no_supervisado' %}
X = preprocess_data(df)
print('X shape:', X.shape)
{% endif %}


{% if ml_type == 'redes_neuronales' %}
### 3b. Análisis del tensor de entrada para `{{ nn_model }}`

PyTorch requiere que los datos sean tensores con la forma correcta según la arquitectura.  
Verifica a continuación que `input_dim` y `output_dim` son consistentes con tus datos:
{% endif %}

In [ ]:
{% if ml_type == 'redes_neuronales' %}
import torch
import numpy as np

INPUT_DIM  = X_train.shape[1]
OUTPUT_DIM = int(y_train.nunique())

print(f"{'input_dim':<15}: {INPUT_DIM}  (features por muestra)")
print(f"{'output_dim':<15}: {OUTPUT_DIM}  ({'clases' if OUTPUT_DIM > 1 else 'salida escalar'})")
print(f"{'Muestras train':<15}: {X_train.shape[0]:,}")
print(f"{'Muestras test':<15}: {X_test.shape[0]:,}")

{% if nn_model == 'MLP' %}
# MLP: espera tensores 2D (batch, input_dim) — sin preproceso extra
x_sample = torch.tensor(X_train[:4].values, dtype=torch.float32)
print(f"\nForma tensor de entrada MLP: {x_sample.shape}  → (batch, input_dim)")

{% elif nn_model == 'CNN1D' %}
# CNN1D: internamente añade dimensión de canal (batch, 1, input_dim)
x_sample = torch.tensor(X_train[:4].values, dtype=torch.float32)
x_conv = x_sample.unsqueeze(1)  # añade dim de canal
print(f"\nForma tensor raw:          {x_sample.shape}")
print(f"Forma para Conv1d (batch, 1, L): {x_conv.shape}")

{% elif nn_model in ['LSTM', 'GRU'] %}
# LSTM/GRU: espera (batch, seq_len, input_size)
# Si los datos son tabulares, se tratan como secuencia de longitud 1
x_sample = torch.tensor(X_train[:4].values, dtype=torch.float32).unsqueeze(1)
print(f"\nForma tensor para {{ nn_model }}: {x_sample.shape}  → (batch, seq_len=1, input_dim)")
print("Consejo: si tus datos tienen estructura temporal real,")
print("  reordena las features en bloques de tiempo antes de entrenar.")

{% elif nn_model == 'Transformer' %}
# Transformer: espera (seq_len, batch, d_model)
x_sample = torch.tensor(X_train[:4].values, dtype=torch.float32).unsqueeze(0)
print(f"\nForma tensor para Transformer: {x_sample.shape}  → (seq_len=1, batch, d_model=input_dim)")
{% endif %}

# Verificar que no hay NaNs ni infs (PyTorch no lo detecta en tiempo de entrenamiento)
X_arr = X_train.values.astype('float32')
print(f"\nNaNs en X_train: {np.isnan(X_arr).sum()}")
print(f"Infs en X_train: {np.isinf(X_arr).sum()}")
if np.isnan(X_arr).any() or np.isinf(X_arr).any():
    print("  Hay valores problemáticos — revisa el preprocesado en build_features.py")
else:
    print("✓ Sin NaNs ni infs — el tensor está listo para entrenamiento")
{% endif %}

{% if ml_type == 'redes_neuronales' %}
### 3c. Tipos de datos y balance de clases

{% if task_type == 'clasificacion' %}
Para `CrossEntropyLoss`, las etiquetas deben ser `torch.long` (enteros).  
Para `BCEWithLogitsLoss`, las etiquetas deben ser `torch.float32`.
{% else %}
Para `MSELoss` / `L1Loss`, los targets deben ser `torch.float32`.
{% endif %}
{% endif %}

In [ ]:
{% if ml_type == 'redes_neuronales' %}
import pandas as pd

# Resumen de dtype y estadísticas del target
{% if task_type == 'clasificacion' %}
print("Distribución de clases (train):")
print(y_train.value_counts().sort_index().to_string())
print()
vc = y_train.value_counts(normalize=True)
imbalance = vc.max() / vc.min()
if imbalance > 3:
    print(f"  Desbalance detectado ({imbalance:.1f}x) — considera class_weight o oversampling")
else:
    print(f"✓ Clases razonablemente balanceadas (ratio max/min: {imbalance:.2f}x)")

# Las etiquetas deben ser enteros contiguos desde 0 para CrossEntropyLoss
classes = sorted(y_train.unique())
if classes != list(range(len(classes))):
    print(f"\n  Las clases {classes} no son enteros desde 0.")
    print("   Usa LabelEncoder en build_features.py antes de guardar y_train.csv")
else:
    print(f"\n✓ Clases: {classes}  → formato correcto para CrossEntropyLoss (torch.long)")
{% else %}
print("Estadísticas del target (regresión):")
print(y_train.describe().round(4))
print()
# Estandarización del target es recomendable para regresión profunda
y_mean, y_std = y_train.mean(), y_train.std()
print(f"Media: {y_mean:.4f}  |  Std: {y_std:.4f}")
if y_std > 10:
    print("Consejo: considera escalar el target (y = (y - mean) / std) para mejorar convergencia")
{% endif %}
{% endif %}

## 4. Guardar datos procesados

In [ ]:
{% if ml_type in ['supervisado', 'redes_neuronales'] %}
import pandas as pd
pd.DataFrame(X_train).to_csv(PROCESSED_DATA_DIR / 'X_train.csv', index=False)
pd.DataFrame(X_test).to_csv(PROCESSED_DATA_DIR / 'X_test.csv', index=False)
pd.Series(y_train).to_csv(PROCESSED_DATA_DIR / 'y_train.csv', index=False)
pd.Series(y_test).to_csv(PROCESSED_DATA_DIR / 'y_test.csv', index=False)
{% elif ml_type == 'no_supervisado' %}
import pandas as pd
pd.DataFrame(X).to_csv(PROCESSED_DATA_DIR / 'X_processed.csv', index=False)
{% endif %}
print('Datos guardados en', PROCESSED_DATA_DIR)
